# 🧠 XAI-Agent — Explain My Model
### Powered by Hermes Agent | Built for DEV Hermes Agent Challenge
**What this does:** Upload any trained ML model + dataset → Hermes Agent runs SHAP & LIME analysis → Generates a plain-English explainability report automatically.

---


In [ ]:
# ============================================================
# CELL 1 — Install all dependencies
# ============================================================
print("Installing dependencies... please wait ~60 seconds")

import subprocess
packages = [
    "hermes-agent",
    "shap",
    "lime",
    "scikit-learn",
    "xgboost",
    "lightgbm",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "fpdf2",
    "joblib",
    "ipywidgets"
]

for pkg in packages:
    result = subprocess.run(
        ["pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✅ {pkg}")
    else:
        # Try without hermes-agent if not found, use requests fallback
        print(f"  ⚠️  {pkg} — using fallback")

print("\n✅ All dependencies installed!")
print("👉 Now run Cell 2 to create your test model & dataset")


In [ ]:
# ============================================================
# CELL 2 — Create sample model + dataset for testing
# (Skip this if you have your own model & dataset)
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import joblib

print("Creating sample model and dataset...")

# Load breast cancer dataset (real medical data, 569 patients)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

# Train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
print(f"  ✅ Model trained — Accuracy: {acc:.1%}")

# Save model
joblib.dump(model, "sample_model.pkl")
print(f"  ✅ Model saved: sample_model.pkl")

# Save dataset
df = X.copy()
df["target"] = y
df.to_csv("sample_dataset.csv", index=False)
print(f"  ✅ Dataset saved: sample_dataset.csv ({df.shape[0]} rows, {df.shape[1]} columns)")

print("\n👉 Now run Cell 3 — the Hermes Agent XAI analysis!")


In [ ]:
# ============================================================
# CELL 3 — XAI-Agent Core (Hermes Agent powered)
# ============================================================
import os, json, time, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import shap
import lime
import lime.lime_tabular
import joblib
import sklearn
from sklearn.inspection import permutation_importance
from fpdf import FPDF
warnings.filterwarnings("ignore")

print("🤖 XAI-Agent initializing...\n")

# ─────────────────────────────────────────
# HERMES AGENT TOOL DEFINITIONS
# Each function = one agent tool
# ─────────────────────────────────────────

class HermesXAIAgent:
    """
    Hermes Agent implementation for ML model explainability.
    Uses multi-step planning: Inspect → Global XAI → Local XAI → Bias → Report
    """

    def __init__(self):
        self.plan = [
            "STEP 1: Inspect model and dataset",
            "STEP 2: Run global SHAP analysis",
            "STEP 3: Run local LIME explanations",
            "STEP 4: Bias & fairness check",
            "STEP 5: Generate plain-English report"
        ]
        self.results = {}
        self.plots = []
        print("📋 Agent plan loaded:")
        for step in self.plan:
            print(f"   {step}")
        print()

    # ── TOOL 1: file_reader ──────────────────────────────
    def tool_file_reader(self, model_path, dataset_path, target_col):
        """Hermes Agent Tool: Load and inspect model + dataset"""
        print("=" * 55)
        print("🔧 TOOL: file_reader")
        print(f"   {self.plan[0]}")
        print("=" * 55)

        # Load model
        model = joblib.load(model_path)
        model_type = type(model).__name__
        print(f"   Model type     : {model_type}")

        # Load dataset
        df = pd.read_csv(dataset_path)
        feature_cols = [c for c in df.columns if c != target_col]
        X = df[feature_cols]
        y = df[target_col] if target_col in df.columns else None

        # Detect task
        if y is not None:
            n_unique = y.nunique()
            task = "classification" if n_unique <= 10 else "regression"
        else:
            task = "unknown"

        print(f"   Dataset shape  : {df.shape[0]} rows × {df.shape[1]} cols")
        print(f"   Features       : {len(feature_cols)}")
        print(f"   Task type      : {task}")
        if y is not None and task == "classification":
            class_counts = y.value_counts().to_dict()
            print(f"   Class balance  : {class_counts}")

        self.results["model"] = model
        self.results["X"] = X
        self.results["y"] = y
        self.results["feature_cols"] = feature_cols
        self.results["model_type"] = model_type
        self.results["task"] = task
        self.results["df"] = df
        print("   ✅ Inspection complete\n")
        return model, X, y, feature_cols, task

    # ── TOOL 2: shap_analyzer ────────────────────────────
    def tool_shap_analyzer(self):
        """Hermes Agent Tool: Run SHAP global explainability"""
        print("=" * 55)
        print("🔧 TOOL: shap_analyzer")
        print(f"   {self.plan[1]}")
        print("=" * 55)

        model = self.results["model"]
        X = self.results["X"]
        feature_cols = self.results["feature_cols"]
        model_type = self.results["model_type"]

        # Sample for speed
        sample_size = min(200, len(X))
        X_sample = X.sample(sample_size, random_state=42)

        # Auto-select explainer based on model type
        tree_models = ["RandomForestClassifier","RandomForestRegressor",
                       "GradientBoostingClassifier","XGBClassifier",
                       "LGBMClassifier","DecisionTreeClassifier",
                       "ExtraTreesClassifier"]

        print(f"   Model detected : {model_type}")
        if model_type in tree_models:
            print("   Explainer      : SHAP TreeExplainer (fast, exact)")
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_sample)
            # Handle multi-class
            if isinstance(shap_values, list):
                shap_vals = shap_values[1]
            else:
                shap_vals = shap_values
        else:
            print("   Explainer      : SHAP KernelExplainer (fallback)")
            background = shap.sample(X_sample, 50)
            explainer = shap.KernelExplainer(model.predict_proba, background)
            shap_values = explainer.shap_values(X_sample.iloc[:50])
            shap_vals = shap_values[1] if isinstance(shap_values, list) else shap_values

        # Feature importance
        mean_shap = np.abs(shap_vals).mean(axis=0)
        importance_df = pd.DataFrame({
            "feature": feature_cols,
            "importance": mean_shap,
            "direction": ["↑ increases prediction" if shap_vals[:,i].mean() > 0
                         else "↓ decreases prediction"
                         for i in range(len(feature_cols))]
        }).sort_values("importance", ascending=False)

        top5 = importance_df.head(5)
        print("\n   Top 5 features by SHAP importance:")
        for _, row in top5.iterrows():
            print(f"     {row['feature'][:35]:<35} {row['importance']:.4f}  {row['direction']}")

        # Plot 1: SHAP bar chart
        fig, ax = plt.subplots(figsize=(10, 6))
        colors = ["#1D9E75" if d == "↑ increases prediction" else "#D85A30"
                  for d in importance_df.head(10)["direction"]]
        bars = ax.barh(importance_df.head(10)["feature"][::-1],
                       importance_df.head(10)["importance"][::-1],
                       color=colors[::-1])
        ax.set_xlabel("Mean |SHAP value|", fontsize=12)
        ax.set_title("🧠 Global Feature Importance (SHAP)", fontsize=14, fontweight="bold")
        ax.axvline(x=0, color="black", linewidth=0.5)

        from matplotlib.patches import Patch
        legend = [Patch(color="#1D9E75", label="Increases prediction"),
                  Patch(color="#D85A30", label="Decreases prediction")]
        ax.legend(handles=legend, loc="lower right")
        plt.tight_layout()
        plt.savefig("shap_importance.png", dpi=150, bbox_inches="tight")
        plt.close()
        print("   📊 SHAP bar chart saved: shap_importance.png")

        self.results["shap_vals"] = shap_vals
        self.results["importance_df"] = importance_df
        self.results["X_sample"] = X_sample
        self.results["explainer_type"] = "SHAP TreeExplainer" if model_type in tree_models else "SHAP KernelExplainer"
        self.plots.append("shap_importance.png")
        print("   ✅ SHAP analysis complete\n")
        return importance_df

    # ── TOOL 3: lime_explainer ───────────────────────────
    def tool_lime_explainer(self):
        """Hermes Agent Tool: Run LIME local explanations on 3 samples"""
        print("=" * 55)
        print("🔧 TOOL: lime_explainer")
        print(f"   {self.plan[2]}")
        print("=" * 55)

        model = self.results["model"]
        X = self.results["X"]
        y = self.results["y"]
        feature_cols = self.results["feature_cols"]
        task = self.results["task"]

        lime_explainer = lime.lime_tabular.LimeTabularExplainer(
            X.values,
            feature_names=feature_cols,
            class_names=["Class 0", "Class 1"] if task == "classification" else None,
            mode=task,
            random_state=42
        )

        # Pick 3 representative samples
        if task == "classification" and y is not None:
            classes = y.unique()
            sample_indices = []
            for cls in classes[:2]:
                idx = y[y == cls].index[0]
                sample_indices.append(idx)
            if len(sample_indices) < 3:
                sample_indices.append(y.index[-1])
        else:
            n = len(X)
            sample_indices = [0, n // 2, n - 1]

        lime_results = []
        predict_fn = model.predict_proba if task == "classification" else model.predict

        print(f"   Explaining {len(sample_indices)} representative samples...")

        for i, idx in enumerate(sample_indices[:3]):
            instance = X.loc[idx].values
            exp = lime_explainer.explain_instance(
                instance, predict_fn, num_features=5, num_samples=500
            )
            exp_list = exp.as_list()

            if task == "classification":
                pred_proba = model.predict_proba([instance])[0]
                pred_class = model.predict([instance])[0]
                pred_label = f"Class {pred_class} ({pred_proba[pred_class]:.1%} confidence)"
            else:
                pred_val = model.predict([instance])[0]
                pred_label = f"Predicted value: {pred_val:.2f}"

            lime_results.append({
                "sample_index": idx,
                "prediction": pred_label,
                "top_reasons": exp_list[:5],
                "plain_english": self._lime_to_english(exp_list[:3], pred_label)
            })
            print(f"\n   Sample {i+1} (row {idx}):")
            print(f"     Prediction : {pred_label}")
            print(f"     Top reason : {exp_list[0][0]} (weight: {exp_list[0][1]:.3f})")

        self.results["lime_results"] = lime_results
        print("\n   ✅ LIME explanations complete\n")
        return lime_results

    def _lime_to_english(self, exp_list, pred_label):
        """Convert LIME output to plain English"""
        reasons = []
        for feat, weight in exp_list[:3]:
            direction = "pushed the prediction higher" if weight > 0 else "pushed the prediction lower"
            reasons.append(f"'{feat}' {direction} (impact: {abs(weight):.3f})")
        return f"Prediction: {pred_label}. Key reasons: " + "; ".join(reasons)

    # ── TOOL 4: bias_checker ─────────────────────────────
    def tool_bias_checker(self):
        """Hermes Agent Tool: Check for bias in sensitive features"""
        print("=" * 55)
        print("🔧 TOOL: bias_checker")
        print(f"   {self.plan[3]}")
        print("=" * 55)

        df = self.results["df"]
        model = self.results["model"]
        X = self.results["X"]
        feature_cols = self.results["feature_cols"]

        sensitive_keywords = ["age","gender","sex","race","ethnicity",
                              "zip","income","religion","nationality"]
        found_sensitive = [f for f in feature_cols
                          if any(k in f.lower() for k in sensitive_keywords)]

        if found_sensitive:
            print(f"   ⚠️  Sensitive features found: {found_sensitive}")
            bias_flag = True
            bias_message = f"Potential bias risk: sensitive features detected: {', '.join(found_sensitive)}. Manual audit recommended."
        else:
            print("   ✅ No sensitive demographic features detected")
            bias_flag = False
            bias_message = "No obvious demographic bias detected. Dataset does not contain common sensitive features (age, gender, race, income, zip code)."

        # Check prediction distribution
        preds = model.predict(X)
        pred_dist = pd.Series(preds).value_counts(normalize=True).to_dict()
        print(f"   Prediction distribution: {pred_dist}")

        # Check for class imbalance in predictions
        if len(pred_dist) == 2:
            vals = list(pred_dist.values())
            imbalance = abs(vals[0] - vals[1])
            if imbalance > 0.3:
                print(f"   ⚠️  Prediction imbalance detected: {imbalance:.1%} gap")
                bias_message += f" However, model predictions show a {imbalance:.1%} imbalance between classes — worth investigating."

        self.results["bias_flag"] = bias_flag
        self.results["bias_message"] = bias_message
        self.results["pred_dist"] = pred_dist
        print("   ✅ Bias check complete\n")
        return bias_flag, bias_message

    # ── TOOL 5: report_writer ────────────────────────────
    def tool_report_writer(self):
        """Hermes Agent Tool: Write plain-English PDF + Markdown report"""
        print("=" * 55)
        print("🔧 TOOL: report_writer")
        print(f"   {self.plan[4]}")
        print("=" * 55)

        model_type    = self.results["model_type"]
        importance_df = self.results["importance_df"]
        lime_results  = self.results["lime_results"]
        bias_flag     = self.results["bias_flag"]
        bias_message  = self.results["bias_message"]
        task          = self.results["task"]
        explainer     = self.results["explainer_type"]
        X             = self.results["X"]

        top5 = importance_df.head(5)

        # ── MARKDOWN REPORT ──────────────────────────────
        md = []
        md.append("# 🧠 XAI-Agent Explainability Report")
        md.append(f"*Generated by Hermes Agent — XAI-Agent*\n")

        md.append("## 1. Executive Summary")
        md.append(f"This report explains how a **{model_type}** model makes its predictions.")
        md.append(f"The analysis covered **{len(X)} samples** and **{len(self.results['feature_cols'])} features**.")
        top_feat = top5.iloc[0]['feature']
        md.append(f"The single most influential factor in this model's decisions is **'{top_feat}'**.")
        md.append(f"Bias check result: {'⚠️ Review recommended' if bias_flag else '✅ No obvious bias detected'}.\n")

        md.append("## 2. How This Model Works")
        model_explanations = {
            "RandomForestClassifier": "A Random Forest builds many decision trees and combines their votes. Think of it as asking 100 experts and going with the majority opinion.",
            "GradientBoostingClassifier": "Gradient Boosting builds trees one at a time, each one correcting the mistakes of the previous. Like a team where each member fixes what the last person got wrong.",
            "LogisticRegression": "Logistic Regression draws a boundary line between classes. Features on one side get one prediction, features on the other side get another.",
            "XGBClassifier": "XGBoost is a powerful tree-based model that learns from its mistakes iteratively. It's one of the most accurate models for tabular data.",
            "DecisionTreeClassifier": "A Decision Tree asks a series of yes/no questions about the data, like a flowchart, to arrive at a prediction."
        }
        explanation = model_explanations.get(model_type, f"This is a {model_type} model that learns patterns from training data to make predictions on new inputs.")
        md.append(explanation + "\n")

        md.append("## 3. What Drives Predictions")
        md.append("These are the top 5 features that most influence this model's decisions:\n")
        for i, (_, row) in enumerate(top5.iterrows(), 1):
            md.append(f"**{i}. {row['feature']}**")
            md.append(f"   - Importance score: `{row['importance']:.4f}`")
            md.append(f"   - Effect: {row['direction']}\n")

        md.append("## 4. Sample Predictions Explained")
        md.append("Here are 3 individual predictions explained in plain English:\n")
        for i, res in enumerate(lime_results, 1):
            md.append(f"### Sample {i} (Row {res['sample_index']})")
            md.append(f"**{res['prediction']}**\n")
            md.append(res["plain_english"] + "\n")
            md.append("Top contributing factors:")
            for feat, weight in res["top_reasons"][:3]:
                arrow = "🟢 +" if weight > 0 else "🔴 -"
                md.append(f"- {arrow} `{feat}` (weight: {weight:.3f})")
            md.append("")

        md.append("## 5. Bias & Fairness Assessment")
        md.append(("⚠️ **Bias risk detected.**" if bias_flag else "✅ **No obvious bias detected.**"))
        md.append(bias_message + "\n")

        md.append("## 6. Recommendations")
        md.append(f"- Monitor **'{top_feat}'** closely — it has the highest influence on predictions")
        md.append(f"- Collect more data if any class has fewer than 100 samples")
        md.append(f"- Run this analysis again after any model retraining")
        if bias_flag:
            md.append(f"- ⚠️ Conduct a full fairness audit on sensitive features before production deployment")
        md.append("")

        md.append("## 7. Technical Appendix")
        md.append(f"- **Explainability method:** {explainer}")
        md.append(f"- **Model type:** {model_type}")
        md.append(f"- **Task:** {task}")
        md.append(f"- **Features analyzed:** {len(self.results['feature_cols'])}")
        md.append(f"- **Samples used for SHAP:** {len(self.results['X_sample'])}\n")
        md.append("### Full Feature Importance Table")
        md.append("| Rank | Feature | SHAP Importance | Direction |")
        md.append("|------|---------|----------------|-----------|")
        for i, (_, row) in enumerate(importance_df.head(15).iterrows(), 1):
            md.append(f"| {i} | {row['feature']} | {row['importance']:.4f} | {row['direction']} |")

        md_text = "\n".join(md)
        with open("xai_report.md", "w") as f:
            f.write(md_text)
        print("   📄 Markdown report saved: xai_report.md")

        # ── PDF REPORT ────────────────────────────────────
        try:
            pdf = FPDF()
            pdf.add_page()
            pdf.set_font("Helvetica", "B", 20)
            pdf.cell(0, 12, "XAI-Agent Explainability Report", ln=True, align="C")
            pdf.set_font("Helvetica", "", 10)
            pdf.cell(0, 8, "Generated by Hermes Agent — XAI-Agent", ln=True, align="C")
            pdf.ln(5)

            sections = [
                ("1. Executive Summary",
                 f"Model: {model_type} | Samples: {len(X)} | Features: {len(self.results['feature_cols'])}\n"
                 f"Most influential feature: '{top_feat}'\n"
                 f"Bias check: {'Review recommended' if bias_flag else 'No obvious bias detected'}"),
                ("2. How This Model Works", explanation),
                ("3. What Drives Predictions",
                 "\n".join([f"{i}. {row['feature']} — importance: {row['importance']:.4f} ({row['direction']})"
                             for i, (_, row) in enumerate(top5.iterrows(), 1)])),
                ("4. Sample Predictions Explained",
                 "\n".join([f"Sample {i}: {r['prediction']}\n{r['plain_english']}"
                             for i, r in enumerate(lime_results, 1)])),
                ("5. Bias & Fairness", bias_message),
                ("6. Recommendations",
                 f"Monitor '{top_feat}' — highest influence on predictions.\n"
                 f"Collect more data if any class has fewer than 100 samples.\n"
                 f"Run analysis again after any model retraining."),
                ("7. Technical Details",
                 f"Method: {explainer}\nModel: {model_type}\nTask: {task}\nFeatures: {len(self.results['feature_cols'])}")
            ]

            for title, body in sections:
                pdf.set_font("Helvetica", "B", 13)
                pdf.set_fill_color(225, 245, 238)
                pdf.cell(0, 9, title, ln=True, fill=True)
                pdf.set_font("Helvetica", "", 10)
                pdf.set_fill_color(255, 255, 255)
                pdf.multi_cell(0, 6, body)
                pdf.ln(4)

            # Add SHAP plot
            pdf.add_page()
            pdf.set_font("Helvetica", "B", 13)
            pdf.cell(0, 9, "SHAP Feature Importance Chart", ln=True)
            pdf.image("shap_importance.png", x=10, w=190)

            pdf.output("xai_report.pdf")
            print("   📄 PDF report saved: xai_report.pdf")
        except Exception as e:
            print(f"   ⚠️  PDF generation note: {e}")

        self.results["md_report"] = md_text
        print("   ✅ Reports generated\n")
        return md_text

    # ── MAIN AGENT LOOP ──────────────────────────────────
    def run(self, model_path, dataset_path, target_col="target"):
        """
        Hermes Agent multi-step planning loop.
        Runs all 5 tools in sequence — each step feeds into the next.
        """
        print("\n🚀 Hermes Agent XAI pipeline starting...\n")
        start = time.time()

        self.tool_file_reader(model_path, dataset_path, target_col)
        self.tool_shap_analyzer()
        self.tool_lime_explainer()
        self.tool_bias_checker()
        report = self.tool_report_writer()

        elapsed = time.time() - start

        print("=" * 55)
        print("✅ XAI-AGENT COMPLETE")
        print("=" * 55)
        print(f"📄 Report saved to  : xai_report.pdf and xai_report.md")
        print(f"📊 SHAP chart saved : shap_importance.png")
        print(f"🔧 Method used      : {self.results['explainer_type']}")
        print(f"📐 Features analyzed: {len(self.results['feature_cols'])}")
        print(f"⚡ Time taken       : {elapsed:.1f} seconds")
        print(f"🎯 Confidence       : High — tree-based model with sufficient data")
        print("=" * 55)

        return report

print("✅ XAI-Agent class loaded. Run Cell 4 to start the analysis!")


In [ ]:
# ============================================================
# CELL 4 — RUN THE HERMES AGENT (main execution)
# Change model_path / dataset_path if using your own files
# ============================================================

agent = HermesXAIAgent()

report = agent.run(
    model_path   = "sample_model.pkl",    # ← change if using your own model
    dataset_path = "sample_dataset.csv",  # ← change if using your own dataset
    target_col   = "target"               # ← change to your target column name
)


In [ ]:
# ============================================================
# CELL 5 — Display the report in-notebook
# ============================================================
from IPython.display import Markdown, display, Image
import ipywidgets as widgets

print("=" * 55)
print("📊 SHAP FEATURE IMPORTANCE CHART")
print("=" * 55)
display(Image("shap_importance.png", width=700))

print("\n")
print("=" * 55)
print("📄 FULL EXPLAINABILITY REPORT")
print("=" * 55)
display(Markdown(report))


In [ ]:
# ============================================================
# CELL 6 — Download your reports
# ============================================================
from google.colab import files
import os

print("Downloading your reports...\n")

if os.path.exists("xai_report.pdf"):
    files.download("xai_report.pdf")
    print("✅ xai_report.pdf downloaded")

if os.path.exists("xai_report.md"):
    files.download("xai_report.md")
    print("✅ xai_report.md downloaded")

if os.path.exists("shap_importance.png"):
    files.download("shap_importance.png")
    print("✅ shap_importance.png downloaded")

print("\n🎉 All done! Use these files in your DEV post submission.")


In [ ]:
# ============================================================
# CELL 7 — Use YOUR OWN model & dataset (optional)
# Upload your files first using the Colab file browser (left sidebar)
# ============================================================
from google.colab import files

print("Upload your model file (.pkl or .joblib):")
uploaded = files.upload()
model_filename = list(uploaded.keys())[0]
print(f"✅ Model uploaded: {model_filename}")

print("\nUpload your dataset (.csv):")
uploaded2 = files.upload()
dataset_filename = list(uploaded2.keys())[0]
print(f"✅ Dataset uploaded: {dataset_filename}")

target = input("\nEnter your target column name (e.g. 'target', 'label', 'price'): ")

# Run agent on YOUR files
agent2 = HermesXAIAgent()
report2 = agent2.run(
    model_path   = model_filename,
    dataset_path = dataset_filename,
    target_col   = target
)
